# Adaptive Flashcard Quiz Agent

A small, repeatable CA-1 demonstration using the same `agent.py`, `tools.py`, and in-memory state as the Streamlit application. It deliberately uses the Normal Quiz path, so every cell runs without a Groq API key and no credential is displayed or hardcoded.

In [ ]:
from agent import run_agent
from memory import create_memory, memory_summary

# Fresh session memory: the same shape used by the Streamlit session.
memory = create_memory()

def show_execution(label, response):
    """Display the trace produced by the shared agent, without recreating its logic."""
    print(f'\n--- {label} ---')
    for item in response['trace']:
        print(f"{item['step']}. {item['title']}: {item['content']}")
    # This is the real result observed after the agent's tool/action.
    print('OBSERVATION (tool result):', response['result'])
    print('FINAL RESPONSE:', response['final_response'])
    return response


## 1. Add multiple flashcards through the shared agent

This uses the production `run_agent()` entry point. Its deterministic manual-flashcard path calls the production `add_card()` tool without requiring Groq, a network connection, or an API key.

In [ ]:
flashcards = [
    ('What is Python?', 'A programming language'),
    ('What does OOP stand for?', 'Object-Oriented Programming'),
    ('What is an API?', 'Application Programming Interface'),
]

manual_addition = show_execution(
    'Add manual flashcards offline',
    run_agent('Add flashcard:\n' + '\n'.join(f'{question} -> {answer}' for question, answer in flashcards), memory),
)
assert manual_addition['result']['status'] == 'success'
assert [item['card_id'] for item in manual_addition['result']['added']] == [1, 2, 3]
print('Memory now contains:', memory['flashcards'])


## 2. Normal Quiz: one correct and one incorrect answer

These calls use the shared `run_agent()` entry point. The printed records are its actual trace: Memory Read → Plan → Tool Decision/Tool Call → Observation → Memory Update → Next Action Decision → Final Response.

In [ ]:
first_quiz = show_execution('Start Normal Quiz', run_agent('Start a Normal Quiz', memory))
assert first_quiz['result']['card_id'] == 1

first_answer = show_execution(
    'Submit a correct answer',
    run_agent('My answer is ready', memory, answer='A programming language'),
)
assert first_answer['result']['correct'] is True

second_quiz = show_execution('Start another Normal Quiz', run_agent('Start a Normal Quiz', memory))
assert second_quiz['result']['card_id'] == 2

second_answer = show_execution(
    'Submit an incorrect answer',
    run_agent('My answer is ready', memory, answer='A programming paradigm'),
)
assert second_answer['result']['correct'] is False

# The next question is different because a completed card is not repeated
# immediately when another eligible card exists.
intermediate_quiz = show_execution('Start the next Normal Quiz', run_agent('Start a Normal Quiz', memory))
assert intermediate_quiz['result']['card_id'] == 3
intermediate_answer = show_execution(
    'Submit a correct answer to the intervening card',
    run_agent('My answer is ready', memory, answer='Application Programming Interface'),
)
assert intermediate_answer['result']['correct'] is True


## 3. Inspect persisted in-session performance

The displayed values come from the same memory dictionary updated by `evaluate_answer()` through the agent.

In [ ]:
summary = memory_summary(memory)
print('Card 1 performance:', memory['performance']['1'])
print('Card 2 performance:', memory['performance']['2'])
print('Custom flashcards topic performance:', memory['topic_performance']['Custom flashcards'])
print('Summary:', {key: summary[key] for key in ('correct_answers', 'incorrect_answers', 'accuracy_percentage', 'weak_topics')})

assert memory['performance']['1'] == {'correct': 1, 'wrong': 0, 'weakness_score': 0}
assert memory['performance']['2'] == {'correct': 0, 'wrong': 1, 'weakness_score': 1}


## 4. Quiz me again: stored weakness prioritises the weak card

Card 2 was answered incorrectly. After card 3 intervened, `quiz_me()` should select card 2 again despite its recent-card penalty.

In [ ]:
retry_quiz = show_execution('Quiz me again', run_agent('Quiz me again: start a Normal Quiz', memory))
print('Stored-memory selection reason:', retry_quiz['result']['selection_reason'])

assert retry_quiz['result']['card_id'] == 2
assert 'Highest weakness score: 1' in retry_quiz['result']['selection_reason']
print('Proof: weak card 2 was prioritised from the shared memory.')
